In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('./Police_Department_Merged_Personal_Focus_Crimes.csv', low_memory=False)

# Parse datetime
df['Incident_Date'] = pd.to_datetime(df['Incident_Date'], errors='coerce')
df['Year'] = df['Incident_Date'].dt.year

# Filter for Larceny/Theft
df_theft = df[df['Crime_Type'] == 'Larceny/Theft']

# Yearly counts
yearly_counts = df_theft.groupby('Year').size()

# Plot
plt.figure(figsize=(10,6))
plt.plot(yearly_counts.index, yearly_counts.values, marker='o')

plt.title('Larceny/Theft in San Francisco (2003–2025)')
plt.xlabel('Year')
plt.ylabel('Number of Incidents')

plt.grid(alpha=0.3)

# Save
plt.savefig('larceny_time_series.png', dpi=300, bbox_inches='tight')
plt.close()

In [3]:
import folium
from folium.plugins import HeatMap

# Drop missing coordinates and work on a copy
df_map = df_theft.dropna(subset=['Latitude', 'Longitude']).copy()

# Convert coordinates to numeric floats (invalid values become NaN)
df_map['Latitude'] = pd.to_numeric(df_map['Latitude'], errors='coerce')
df_map['Longitude'] = pd.to_numeric(df_map['Longitude'], errors='coerce')

# Keep only valid coordinate rows after conversion
df_map = df_map.dropna(subset=['Latitude', 'Longitude'])

# Create base map (SF center)
m = folium.Map(location=[37.77, -122.42], zoom_start=12)

# Add heatmap
heat_data = df_map[['Latitude', 'Longitude']].values.tolist()

HeatMap(heat_data, radius=8).add_to(m)

# Save
m.save('larceny_heatmap.html')

In [ ]:
import plotly.express as px

# Extract hour + year
df_theft['Hour'] = df_theft['Incident_Time'].dt.hour

# Group
yearly_hourly = (
    df_theft.groupby(['Year', 'Hour'])
    .size()
    .reset_index(name='Count')
)

# Normalize per year
yearly_hourly['NormalizedCount'] = (
    yearly_hourly['Count'] /
    yearly_hourly.groupby('Year')['Count'].transform('sum')
)

# Plot
fig = px.line(
    yearly_hourly,
    x='Hour',
    y='NormalizedCount',
    animation_frame='Year',
    title='Hourly Pattern of Larceny/Theft Over Time',
    range_y=[0, 0.15]
)

# Improve layout
fig.update_layout(
    xaxis=dict(tickmode='linear', range=[0,23]),
    yaxis_title='Probability of Theft',
    xaxis_title='Hour of Day'
)

# Save
fig.write_html('larceny_hourly_animation.html')

AttributeError: Can only use .dt accessor with datetimelike values